In [ ]:
#
# Import Data
#

import rasterio
import os
import geopandas as gpd
import xarray as xr
import rioxarray
import numpy as np
import matplotlib.pyplot as plt
import gdown
from scipy.stats import linregress
import cmcrameri.cm as cmc
from matplotlib.colors import LightSource

# Create data directory
data_folder = "data/raw"
os.makedirs(data_folder, exist_ok=True)

filepath = "data/raw/LandsatComposite_Zurich_1985.tif"

# Open the raster connection safely
#with rasterio.open(filepath) as src:
	#print(src.name)
	#print(src.mode)
	#print("Band count:", src.count)

# Bands
# Blue, Green, Red, NIR, SWIR1, SWIR2, TIR

# Load test data cube
#ds = xr.tutorial.open_dataset("air_temperature")
#print(ds)

# Loop over years to create filepaths filepaths
years = [1985, 1988, 1991, 1994, 1997, 2000, 2003, 2006, 2009, 2012, 2015, 2018, 2021, 2024]
ls_paths = []

for year in years:
	fp = "data/raw/LandsatComposite_Zurich_" + str(year) + ".tif"

	ls_paths.append(fp)

#print(ls_paths)

# Loop over filepaths to import data into data cube
bands = ["Blue", "Green", "Red", "NIR", "SWIR1", "SWIR2", "TIR"]
list_da = []

for path in ls_paths:
	da = rioxarray.open_rasterio(path)

	dt = path.split("_")[2].split(".")[0]

	da = da.assign_coords(band = bands)
	da = da.expand_dims(time = [dt])
	
	list_da.append(da)

ls_cube = xr.combine_by_coords(list_da)

#ls_cube

In [ ]:
#
# Visualise RGB image
#

import numpy as np
import rasterio
import matplotlib.pyplot as plt

with rasterio.open(filepath) as src:
	red = src.read(3)
	green = src.read(2)
	blue = src.read(1)


def normalize(array, vmin=0, vmax=0.4):
	"""Normalize and clip array to a specific range (default 0 to 0.4)"""
	# Clip the array to the specified vmin and vmax
	clipped = np.clip(array, vmin, vmax)
	# Scale the clipped array to 0-1 for display
	return (clipped - vmin) / (vmax - vmin)


red_n = normalize(red)
green_n = normalize(green)
blue_n = normalize(blue)

# Stack the normalized bands along a third dimension
rgb = np.dstack((red_n, green_n, blue_n))

plt.figure(figsize=(7, 6))
plt.imshow(rgb)
plt.title("RGB true color composite (Scaled 0-0.4)")
plt.axis("off")
plt.show()

In [ ]:
#
# Visualise IR Bands
#

import matplotlib.pyplot as plt
from rasterio.plot import show

with rasterio.open(filepath) as src:
	fig, (ax4, ax5, ax6, ax7) = plt.subplots(ncols=4, figsize=(11, 4), sharey=True)
#, ax4, ax5, ax6, ax7
	#show((src, 1), cmap="viridis", ax=ax1)
	#show((src, 2), cmap="viridis", ax=ax2)
	#show((src, 3), cmap="viridis", ax=ax3)
	show((src, 4), cmap="viridis", ax=ax4)
	show((src, 5), cmap="viridis", ax=ax5)
	show((src, 6), cmap="viridis", ax=ax6)
	show((src, 7), cmap="viridis", ax=ax7)

	#ax1.set_title("Band 1 (Blue)")
	#ax2.set_title("Band 2 (Green)")
	#ax3.set_title("Band 3 (Red)")
	ax4.set_title("Band 4 (NIR)")
	ax5.set_title("Band 5 (SWIR1)")
	ax6.set_title("Band 6 (SWIR2)")
	ax7.set_title("Band 7 (TIR)")
	
	#ax1.axis("off")
	#ax2.axis("off")
	#ax3.axis("off")
	ax4.axis("off")
	ax5.axis("off")
	ax6.axis("off")
	ax7.axis("off")

plt.show()

In [ ]:
#
# Calculate NDVI
#

import rasterio
import numpy as np
import matplotlib.pyplot as plt

with rasterio.open(filepath) as src:
	# Read the bands and immediately convert them to float to allow division
	# Band 3 = Red, Band 4 = NIR
	red = src.read(3).astype(float)
	nir = src.read(4).astype(float)

# Suppress warnings for zero division and invalid operations
with np.errstate(divide="ignore", invalid="ignore"):
	# Perform the pixel by pixel array math
	ndvi = (nir - red) / (nir + red)

# Plot the calculated index
plt.figure(figsize=(7, 6))
plt.imshow(ndvi)
plt.title("NDVI")
plt.axis("off")
plt.show()